# Baseline Model: 뉴스 기사 → 온톨로지 기반 지식그래프

이 노트북은 `baseline/deep-research-report.md`의 설계를 실행 가능한 최소 기준선으로 옮긴 것이다. 베이스 모델이므로 LLM, Deep Agent, Neo4j, SHACL 같은 운영 의존성은 쓰지 않고, 같은 흐름을 **규칙 기반 파서와 allowlist 검증**으로 재현한다.

핵심 목적은 세 가지다.

- 결정론적 파싱, 구조화 추출, 온톨로지 매핑, 검증 게이트, provenance 기록의 흐름을 작게 고정한다.
- 이후 GLiNER, REBEL/mREBEL, LLM structured output, Deep Agent coordinator로 교체할 때 비교할 기준선을 만든다.
- 모델 성능보다 중요한 운영 조건인 evidence, schema, ontology allowlist, idempotent graph payload를 코드에서 명확히 보인다.


## 1. 기준선 설계

보고서는 Deep Agent를 추출기 자체가 아니라 **긴 작업을 조정하고 검증하는 오케스트레이터**로 두는 방식을 권장한다. 이 베이스라인은 오케스트레이션 없이도 실제 파이프라인의 핵심 산출물을 확인할 수 있게 만든다.

1. 기사 원문을 canonical article object로 만든다.
2. 문장 chunk를 만들고 evidence 위치를 보존한다.
3. alias와 타입별 규칙으로 객체를 탐지한다.
4. 온톨로지 domain/range 규칙으로 관계를 탐지한다.
5. 객체 타입별 속성 스키마에 맞춰 속성을 추출한다.
6. 객체, 관계, 속성을 검증하고 Neo4j에 실행 가능한 parameterized Cypher를 생성한다.
7. Cypher가 허용된 label/type과 parameter binding만 쓰는지 별도 게이트로 검사한다.

즉 이 노트북의 기준선은 "모델 성능"보다 **근거 있는 추출 결과가 실제 그래프 DB write 직전까지 갈 수 있는가**를 확인하는 역할이다.


In [13]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import hashlib
import json
import re

# Notebook을 repo root에서 실행해도, baseline 디렉터리에서 실행해도 동작하게 맞춘다.
CWD = Path.cwd()
WORK_DIR = CWD / "baseline" if (CWD / "baseline").exists() else CWD
REPORT_PATH = WORK_DIR / "deep-research-report.md"
OUTPUT_DIR = WORK_DIR / "basemodel_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"WORK_DIR: {WORK_DIR}")
print(f"REPORT_EXISTS: {REPORT_PATH.exists()}")


WORK_DIR: /Users/hyeon/Desktop/Programming/ML_stock_forecast/baseline
REPORT_EXISTS: True


## 2. 데이터 스키마

운영 설계에서는 Pydantic이나 LangChain structured output으로 schema-valid 결과를 받는 것이 좋다. 여기서는 외부 의존성을 줄이기 위해 dataclass를 사용한다. 필드 이름은 보고서의 `EntityMention`, `RelationCandidate`, `AttributeCandidate`와 맞춰 두어 이후 교체 비용을 낮춘다.

**발전 가능성**

- `dataclass`를 Pydantic `BaseModel`로 바꾸면 필수 필드, enum, confidence 범위, 날짜 타입을 자동 검증할 수 있다.
- `EntityMention`, `RelationCandidate`, `AttributeCandidate`를 JSON Schema로 export하면 LLM structured output의 응답 스키마로 그대로 쓸 수 있다.
- `start_char/end_char`만 두는 대신 `EvidenceSpan` 객체를 따로 만들면 page, paragraph, sentence id, token span까지 확장할 수 있다.


In [3]:
@dataclass(frozen=True)
class ArticleDocument:
    article_id: str
    headline: str
    source_url: str
    published_at: str
    text: str
    checksum: str


@dataclass(frozen=True)
class Chunk:
    chunk_id: str
    text: str
    start_char: int
    end_char: int


@dataclass(frozen=True)
class EntityMention:
    mention_text: str
    normalized_name: str
    canonical_id: str
    entity_type: str
    start_char: int
    end_char: int
    evidence_text: str
    chunk_id: str
    confidence: float


@dataclass(frozen=True)
class RelationCandidate:
    subject_id: str
    subject_name: str
    predicate: str
    object_id: str
    object_name: str
    evidence_text: str
    chunk_id: str
    confidence: float


@dataclass(frozen=True)
class AttributeCandidate:
    target_id: str
    target_type: str
    attribute_name: str
    value: Any
    value_text: str
    value_type: str
    evidence_text: str
    chunk_id: str
    confidence: float


## 3. 사전 정의 온톨로지, 속성 스키마, alias KB

베이스라인은 LLM을 쓰지 않지만, 추출 결과가 들어갈 수 있는 공간은 명시적으로 제한한다. 객체 타입, 관계 타입, 타입별 속성 스키마를 먼저 정의하고, 이후 모든 추출 결과는 이 allowlist를 통과해야 한다.

**발전 가능성**

- 현재 dict 기반 온톨로지를 RDF/OWL, JSON-LD, Neo4j constraint, SHACL shape로 옮기면 운영 검증력이 높아진다.
- alias KB는 SKOS `prefLabel/altLabel` 또는 별도 entity linking table로 관리할 수 있다.
- 기업 코드, 종목 티커, ISIN, DART corp code 같은 외부 identifier를 canonical_id와 함께 저장하면 동명이인/동명기업 충돌을 줄일 수 있다.


In [4]:
ONTOLOGY: dict[str, Any] = {
    "entity_types": {
        "NewsArticle": "뉴스 기사",
        "Organization": "기업, 기관, 단체",
        "Person": "사람",
        "Place": "장소 또는 지역",
        "Product": "제품 또는 서비스",
    },
    "relations": {
        "MENTIONS": {
            "domain": ["NewsArticle"],
            "range": ["Organization", "Person", "Place", "Product"],
        },
        "ANNOUNCED": {
            "domain": ["Organization"],
            "range": ["Product"],
            "keywords": ["발표", "공개", "소개"],
        },
        "PARTNERED_WITH": {
            "domain": ["Organization"],
            "range": ["Organization"],
            "keywords": ["협력", "제휴", "파트너십"],
            "symmetric": True,
        },
        "LOCATED_IN": {
            "domain": ["Organization"],
            "range": ["Place"],
            "keywords": ["본사", "위치", "소재"],
        },
        "ANALYZED_BY": {
            "domain": ["Organization"],
            "range": ["Person"],
            "keywords": ["분석가", "언급", "전망"],
        },
    },
}

ATTRIBUTE_SCHEMA: dict[str, dict[str, str]] = {
    "NewsArticle": {
        "mentioned_date": "date",
    },
    "Organization": {
        "market_cap_trillion_krw": "number",
        "revenue_target_trillion_krw": "number",
        "stock_change_percent": "number",
    },
    "Product": {
        "product_category": "string",
    },
    "Person": {
        "role": "string",
    },
    "Place": {
        "admin_area": "string",
    },
}

# 기준선 샘플에서는 기업/제품/사람 객체가 최소한 이 속성을 갖는지 확인한다.
REQUIRED_ATTRIBUTES_BY_TYPE: dict[str, set[str]] = {
    "Organization": {"market_cap_trillion_krw"},
    "Product": {"product_category"},
    "Person": {"role"},
}

ALIAS_KB: dict[str, dict[str, Any]] = {
    "org:samsung-electronics": {
        "name": "삼성전자",
        "entity_type": "Organization",
        "aliases": ["삼성전자", "삼성"],
    },
    "org:sk-hynix": {
        "name": "SK하이닉스",
        "entity_type": "Organization",
        "aliases": ["SK하이닉스", "에스케이하이닉스"],
    },
    "place:seoul": {
        "name": "서울",
        "entity_type": "Place",
        "aliases": ["서울", "서울시", "서초구"],
    },
    "product:exynos-ai": {
        "name": "Exynos AI",
        "entity_type": "Product",
        "aliases": ["Exynos AI", "엑시노스 AI", "차세대 AI 반도체"],
    },
    "person:kim-minsu": {
        "name": "김민수",
        "entity_type": "Person",
        "aliases": ["시장 분석가 김민수", "김민수"],
    },
}

ENTITY_CONFIDENCE_EXACT_ALIAS = 0.95
RELATION_CONFIDENCE_RULE = 0.85
ATTRIBUTE_CONFIDENCE_REGEX = 0.90

print("entity types:", list(ONTOLOGY["entity_types"].keys()))
print("relation types:", list(ONTOLOGY["relations"].keys()))
print("attribute schema:", ATTRIBUTE_SCHEMA)


entity types: ['NewsArticle', 'Organization', 'Person', 'Place', 'Product']
relation types: ['MENTIONS', 'ANNOUNCED', 'PARTNERED_WITH', 'LOCATED_IN', 'ANALYZED_BY']
attribute schema: {'NewsArticle': {'mentioned_date': 'date'}, 'Organization': {'market_cap_trillion_krw': 'number', 'revenue_target_trillion_krw': 'number', 'stock_change_percent': 'number'}, 'Product': {'product_category': 'string'}, 'Person': {'role': 'string'}, 'Place': {'admin_area': 'string'}}


## 4. 샘플 기사와 canonical parsing

샘플 기사는 객체, 관계, 속성이 모두 나오도록 구성했다. 기업에는 시가총액, 제품에는 카테고리, 사람에는 역할이 추출되어야 하며, 이 결과가 나중에 Neo4j node property와 relationship으로 변환된다.

**발전 가능성**

- HTML 뉴스는 `trafilatura`, `newspaper3k`, `BeautifulSoup`, readability 계열 도구로 본문과 메타데이터를 분리할 수 있다.
- PDF/문서 기사에는 Docling 같은 layout-aware parser를 붙이면 표, 캡션, 제목 구조를 더 잘 보존할 수 있다.
- `source_url`, `published_at`, `author`, `publisher`, `section`, `language`, `checksum`을 canonical article object에 넣으면 provenance와 중복 제거가 쉬워진다.


In [5]:
RAW_ARTICLE = {
    "headline": "삼성전자, 서울에서 차세대 AI 반도체 발표",
    "source_url": "https://example.com/news/samsung-ai-chip",
    "published_at": "2026-06-20T09:00:00+09:00",
    "text": (
        "2026년 6월 20일 서울에서 삼성전자는 차세대 AI 반도체 Exynos AI를 발표했다. "
        "Exynos AI는 모바일 AI 반도체 제품으로 소개됐다. "
        "삼성전자의 시가총액은 430조 원이며 SK하이닉스의 시가총액은 190조 원이다. "
        "삼성전자는 2026년 AI 반도체 매출 목표를 25조 원으로 제시했다. "
        "삼성전자는 서울 서초구에 본사를 두고 있다. "
        "삼성전자는 SK하이닉스와 메모리 공급망 협력을 확대한다고 밝혔다. "
        "발표 이후 삼성전자 주가는 3.2% 상승했고, 시장 분석가 김민수는 삼성전자의 수요 회복을 언급했다."
    ),
}


def make_article(raw: dict[str, str]) -> ArticleDocument:
    """원시 기사 dict를 검증하고, headline+본문 앞부분으로 안정적인 article_id를 만든다."""
    required = ["headline", "source_url", "published_at", "text"]
    missing = [field for field in required if not raw.get(field)]
    if missing:
        raise ValueError(f"missing required article fields: {missing}")

    headline = raw["headline"].strip()
    text = raw["text"].strip()
    fingerprint_source = chr(10).join([headline, text[:200]])
    checksum = hashlib.sha256(fingerprint_source.encode("utf-8")).hexdigest()[:16]
    article_id = f"article:{checksum}"
    return ArticleDocument(
        article_id=article_id,
        headline=headline,
        source_url=raw["source_url"].strip(),
        published_at=raw["published_at"].strip(),
        text=text,
        checksum=checksum,
    )


article = make_article(RAW_ARTICLE)
asdict(article)


{'article_id': 'article:3a8a943b5b939909',
 'headline': '삼성전자, 서울에서 차세대 AI 반도체 발표',
 'source_url': 'https://example.com/news/samsung-ai-chip',
 'published_at': '2026-06-20T09:00:00+09:00',
 'text': '2026년 6월 20일 서울에서 삼성전자는 차세대 AI 반도체 Exynos AI를 발표했다. Exynos AI는 모바일 AI 반도체 제품으로 소개됐다. 삼성전자의 시가총액은 430조 원이며 SK하이닉스의 시가총액은 190조 원이다. 삼성전자는 2026년 AI 반도체 매출 목표를 25조 원으로 제시했다. 삼성전자는 서울 서초구에 본사를 두고 있다. 삼성전자는 SK하이닉스와 메모리 공급망 협력을 확대한다고 밝혔다. 발표 이후 삼성전자 주가는 3.2% 상승했고, 시장 분석가 김민수는 삼성전자의 수요 회복을 언급했다.',
 'checksum': '3a8a943b5b939909'}

## LLM

In [12]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "LGAI-EXAONE/EXAONE-4.0-1.2B"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype="bfloat16",
    device_map="auto",
    local_files_only=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    local_files_only=True,
)


Loading weights:   0%|          | 0/332 [00:00<?, ?it/s]

In [7]:
import torch

model.eval()

print("model device:", next(model.parameters()).device)
print("model dtype:", next(model.parameters()).dtype)
print("vocab size:", len(tokenizer))


model device: mps:0
model dtype: torch.bfloat16
vocab size: 102400


In [8]:
prompt = "한국 주식 시장에서 삼성전자 주가에 영향을 줄 수 있는 요인 3가지를 간단히 설명해줘."

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
        temperature=0.1,
        top_p=None,
        pad_token_id=tokenizer.eos_token_id,
    )

print(tokenizer.decode(output[0], skip_special_tokens=True))
print(output)


[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


한국 주식 시장에서 삼성전자 주가에 영향을 줄 수 있는 요인 3가지를 간단히 설명해줘.
tensor([[ 8052,  7558,  2886, 41728, 14118,  9999,  2373,  5282,   696,  1949,
           868,   773,   657, 11737,   582,   380,  6576,  4605, 17300,  3328,
           999, 15887,   375,   361]], device='mps:0')


In [9]:
messages = [
    {"role": "user", "content": "환율 상승이 수출 기업 주가에 미치는 영향을 짧게 설명해줘."}
]

if tokenizer.chat_template:
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
else:
    text = messages[0]["content"]

inputs = tokenizer(text, return_tensors="pt").to(model.device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

print(tokenizer.decode(output[0], skip_special_tokens=True))



환율 상승이 수출 기업 주가에 미치는 영향을 짧게 설명해줘.

<think>

</think>

환율 상승은 수출 기업 주가에 부정적인 영향을 미칩니다. 이유는 다음과 같습니다:  

1. **수출 대금 수령 가치 감소**: 외국에서 받는 달러 등 외화가 더 비싸져 수익성이 떨어집니다.  
2. **원화 약세로 인한 비용 증가**: 현지 생산 비용이 상대적으로 높아져 이익률이 하락할 수 있습니다.  
3. **경쟁 심화**: 환율 상승으로 해외 경쟁사의 수출 가격 경쟁력이 강화될 수 있습니다.  

단, 글로벌 경기 침체 시 환율 상승이 수출 수요 감소를 유발해 추가 타격을 줄 수도 있습니다.


## 5. Chunking

보고서는 긴 기사에서 단계별 추출과 검증을 하기 위해 chunk 단위를 권장한다. 기준선은 문장 단위 chunking을 사용한다. 문장 단위는 관계 evidence를 읽기 쉽고, 잘못된 cross-sentence 추론을 줄인다.

**발전 가능성**

- 한국어 문장 분리는 단순 정규식보다 `kss`, spaCy sentence segmenter, Kiwi 같은 도구를 쓰는 편이 안정적이다.
- 문단 단위 분석이 필요하면 빈 줄(`

`) 기준 paragraph chunk를 먼저 만들고, 너무 긴 문단만 문장 단위로 다시 나누면 된다.
- LangChain `RecursiveCharacterTextSplitter`처럼 max token, overlap, separator 우선순위를 둔 chunking을 쓰면 긴 기사와 LLM 입력에 더 잘 맞는다.


In [ ]:
def split_into_sentence_chunks(article: ArticleDocument) -> list[Chunk]:
    """기사 본문을 문장 단위 chunk로 나누고, 원문 기준 start/end 위치를 함께 보존한다."""
    # 문장 종결 부호 뒤에 공백이나 문자열 끝이 있을 때만 자른다.
    # 이렇게 해야 3.2% 같은 소수점 표현을 문장 끝으로 오인하지 않는다.
    pattern = re.compile(r".+?(?:[.!?。](?=\s|$)|$)", re.DOTALL)
    chunks: list[Chunk] = []

    for match in pattern.finditer(article.text):
        text = match.group(0).strip()
        if not text:
            continue
        chunks.append(
            Chunk(
                chunk_id=f"chunk:{len(chunks) + 1:03d}",
                text=text,
                start_char=match.start(),
                end_char=match.end(),
            )
        )

    if not chunks and article.text:
        chunks.append(Chunk("chunk:001", article.text, 0, len(article.text)))

    return chunks


chunks = split_into_sentence_chunks(article)
for chunk in chunks:
    print(chunk.chunk_id, chunk.start_char, chunk.end_char, chunk.text)


## 6. 객체 추출

객체 추출은 alias exact match를 기본으로 유지하되, 긴 alias 우선 매칭과 span 중복 제거를 사용한다. 이 단계의 목적은 자유 텍스트를 `canonical_id`, `entity_type`, `evidence_text`, `start/end`를 가진 그래프 객체 후보로 바꾸는 것이다.

**발전 가능성**

- spaCy NER를 쓰면 사람, 조직, 장소 같은 기본 엔터티를 모델 기반으로 추출할 수 있고, 프로젝트 도메인에 맞게 fine-tune할 수 있다.
- GLiNER는 런타임에 원하는 label을 넣어 zero/few-shot NER처럼 쓸 수 있어 온톨로지 타입 목록과 잘 맞는다.
- LLM structured output을 붙이면 `canonical_id`, `entity_type`, `confidence`, `evidence_text`를 한 번에 받도록 강제할 수 있다.
- alias exact match 이후 entity linking, coreference resolution을 추가하면 `삼성전자`, `회사`, `동사` 같은 지칭을 같은 객체로 묶을 수 있다.


In [ ]:
def build_alias_patterns(alias_kb: dict[str, dict[str, Any]]) -> list[tuple[str, str, re.Pattern[str]]]:
    """alias 사전을 긴 표현 우선 정규식 목록으로 바꿔 부분 매칭 오류를 줄인다."""
    patterns: list[tuple[str, str, re.Pattern[str]]] = []
    for canonical_id, spec in alias_kb.items():
        for alias in sorted(spec["aliases"], key=len, reverse=True):
            patterns.append((canonical_id, alias, re.compile(re.escape(alias))))
    patterns.sort(key=lambda item: len(item[1]), reverse=True)
    return patterns


def overlaps(existing_spans: list[tuple[int, int]], start: int, end: int) -> bool:
    """새 mention span이 이미 선택된 span과 겹치는지 확인해 중복/부분 매칭을 막는다."""
    return any(start < old_end and old_start < end for old_start, old_end in existing_spans)


def extract_entities(chunks: list[Chunk], alias_kb: dict[str, dict[str, Any]]) -> list[EntityMention]:
    """기사 chunk에서 객체 후보를 추출하고 ontology type과 canonical_id를 부여한다."""
    patterns = build_alias_patterns(alias_kb)
    entities: list[EntityMention] = []
    used_spans: list[tuple[int, int]] = []

    for chunk in chunks:
        for canonical_id, alias, pattern in patterns:
            spec = alias_kb[canonical_id]
            for match in pattern.finditer(chunk.text):
                start = chunk.start_char + match.start()
                end = chunk.start_char + match.end()
                if overlaps(used_spans, start, end):
                    continue
                used_spans.append((start, end))
                entities.append(
                    EntityMention(
                        mention_text=match.group(0),
                        normalized_name=spec["name"],
                        canonical_id=canonical_id,
                        entity_type=spec["entity_type"],
                        start_char=start,
                        end_char=end,
                        evidence_text=chunk.text,
                        chunk_id=chunk.chunk_id,
                        confidence=ENTITY_CONFIDENCE_EXACT_ALIAS,
                    )
                )

    return sorted(entities, key=lambda ent: (ent.start_char, ent.end_char))


entities = extract_entities(chunks, ALIAS_KB)
for ent in entities:
    print(ent)


## 7. 관계 추출

관계 추출은 단순 keyword search가 아니라 **관계 타입별 domain/range + trigger keyword**를 함께 적용한다. 예를 들어 `ANALYZED_BY`는 `Organization -> Person`만 허용되고, `분석가`, `언급` 같은 근거가 있는 chunk에서만 생성된다.

**발전 가능성**

- spaCy dependency parsing을 사용하면 주어-동사-목적어 구조를 보고 관계 방향성을 더 안정적으로 잡을 수 있다.
- REBEL/mREBEL 같은 relation extraction 모델은 문장에서 후보 triple을 생성하는 데 쓸 수 있다.
- LLM structured output으로 후보 관계를 만들되, ontology domain/range와 evidence verifier를 통과한 관계만 유지하는 방식이 안전하다.
- 관계별 rule, classifier, verifier를 분리하면 `후보 생성 -> 온톨로지 매핑 -> 근거 검증` 흐름으로 발전시킬 수 있다.


In [ ]:
def entity_index_by_id(entities: list[EntityMention]) -> dict[str, EntityMention]:
    """canonical_id로 엔터티를 빠르게 찾기 위한 lookup table을 만든다."""
    index: dict[str, EntityMention] = {}
    for ent in entities:
        index.setdefault(ent.canonical_id, ent)
    return index


def entities_in_chunk(chunk: Chunk, entities: list[EntityMention]) -> list[EntityMention]:
    """특정 chunk의 문자 범위 안에 포함된 엔터티 mention만 골라낸다."""
    return [
        ent for ent in entities
        if ent.start_char >= chunk.start_char and ent.end_char <= chunk.end_char
    ]


def type_allowed(entity_type: str, allowed: list[str]) -> bool:
    """엔터티 타입이 온톨로지의 domain/range 허용 목록에 포함되는지 확인한다."""
    return entity_type in allowed


def make_relation(
    subject: EntityMention,
    predicate: str,
    object_: EntityMention,
    chunk: Chunk,
    confidence: float = RELATION_CONFIDENCE_RULE,
) -> RelationCandidate:
    """검증된 subject/object mention과 predicate를 RelationCandidate record로 묶는다."""
    return RelationCandidate(
        subject_id=subject.canonical_id,
        subject_name=subject.normalized_name,
        predicate=predicate,
        object_id=object_.canonical_id,
        object_name=object_.normalized_name,
        evidence_text=chunk.text,
        chunk_id=chunk.chunk_id,
        confidence=confidence,
    )


def extract_relations(chunks: list[Chunk], entities: list[EntityMention], ontology: dict[str, Any]) -> list[RelationCandidate]:
    """관계별 trigger와 ontology domain/range를 모두 만족하는 관계만 추출한다."""
    relations: list[RelationCandidate] = []
    seen: set[tuple[str, str, str, str]] = set()

    for chunk in chunks:
        chunk_entities = entities_in_chunk(chunk, entities)
        for predicate, spec in ontology["relations"].items():
            if predicate == "MENTIONS":
                continue
            if not any(keyword in chunk.text for keyword in spec.get("keywords", [])):
                continue

            subjects = [ent for ent in chunk_entities if type_allowed(ent.entity_type, spec["domain"])]
            objects = [ent for ent in chunk_entities if type_allowed(ent.entity_type, spec["range"])]
            for subject in subjects:
                for object_ in objects:
                    if subject.canonical_id == object_.canonical_id:
                        continue
                    key = (subject.canonical_id, predicate, object_.canonical_id, chunk.chunk_id)
                    reverse_key = (object_.canonical_id, predicate, subject.canonical_id, chunk.chunk_id)
                    if key in seen or (spec.get("symmetric") and reverse_key in seen):
                        continue
                    seen.add(key)
                    relations.append(make_relation(subject, predicate, object_, chunk))

    return relations


relations = extract_relations(chunks, entities, ONTOLOGY)
for rel in relations:
    print(rel)


## 8. 타입별 속성 추출

속성은 객체 타입별 스키마에 맞춰 추출한다. 기업은 `market_cap_trillion_krw`, `revenue_target_trillion_krw`, `stock_change_percent`를 가질 수 있고, 제품은 `product_category`, 사람은 `role`을 갖는다. 모든 속성은 target 객체와 evidence chunk를 함께 가진다.

**발전 가능성**

- 숫자, 날짜, 통화, 퍼센트는 정규식 이후 단위 정규화기를 붙여 `조 원`, `억 원`, `%`, 날짜 포맷을 표준 값으로 바꾸는 것이 좋다.
- `dateparser`, `pandas`, 별도 금융 단위 parser를 사용하면 날짜와 수치 속성의 타입 안정성이 올라간다.
- 역할, 전망, 제품 카테고리처럼 표현이 다양한 속성은 LLM structured output이나 text classification 모델로 보완할 수 있다.
- 속성마다 `value`, `value_text`, `unit`, `currency`, `as_of_date`, `confidence`, `evidence_span`을 분리하면 그래프 분석에 더 유리하다.


In [ ]:
DATE_PATTERN = re.compile(r"\d{4}년\s*\d{1,2}월\s*\d{1,2}일")
MARKET_CAP_PATTERN = re.compile(r"(?P<name>삼성전자|SK하이닉스)의\s*시가총액은\s*(?P<value>\d+(?:\.\d+)?)조\s*원")
REVENUE_TARGET_PATTERN = re.compile(r"(?P<name>삼성전자|SK하이닉스).*?매출 목표를\s*(?P<value>\d+(?:\.\d+)?)조\s*원")
STOCK_CHANGE_PATTERN = re.compile(r"(?P<name>삼성전자|SK하이닉스)\s*주가는\s*(?P<value>\d+(?:\.\d+)?)%\s*(?P<direction>상승|하락)")
ROLE_PATTERN = re.compile(r"(?P<role>시장 분석가)\s*(?P<name>[가-힣]{2,3})(?:는|은|이|가)?")


def find_entity_by_name(name: str, chunk_entities: list[EntityMention], entity_type: str | None = None) -> EntityMention | None:
    """chunk 안에서 표면형 또는 정규화 이름이 일치하는 객체를 찾는다."""
    for ent in chunk_entities:
        if entity_type and ent.entity_type != entity_type:
            continue
        if name in {ent.mention_text, ent.normalized_name}:
            return ent
    return None


def make_attribute(
    target: EntityMention | ArticleDocument,
    target_type: str,
    attribute_name: str,
    value: Any,
    value_text: str,
    value_type: str,
    chunk: Chunk,
    confidence: float = ATTRIBUTE_CONFIDENCE_REGEX,
) -> AttributeCandidate:
    """타입별 속성 추출 결과를 target/evidence와 함께 표준 record로 만든다."""
    target_id = target.article_id if isinstance(target, ArticleDocument) else target.canonical_id
    return AttributeCandidate(
        target_id=target_id,
        target_type=target_type,
        attribute_name=attribute_name,
        value=value,
        value_text=value_text,
        value_type=value_type,
        evidence_text=chunk.text,
        chunk_id=chunk.chunk_id,
        confidence=confidence,
    )


def extract_attributes(article: ArticleDocument, chunks: list[Chunk], entities: list[EntityMention]) -> list[AttributeCandidate]:
    """정규식과 타입별 규칙으로 객체 속성을 추출한다."""
    attributes: list[AttributeCandidate] = []

    for chunk in chunks:
        chunk_entities = entities_in_chunk(chunk, entities)

        for match in DATE_PATTERN.finditer(chunk.text):
            attributes.append(
                make_attribute(article, "NewsArticle", "mentioned_date", match.group(0), match.group(0), "date", chunk)
            )

        for match in MARKET_CAP_PATTERN.finditer(chunk.text):
            target = find_entity_by_name(match.group("name"), chunk_entities, "Organization")
            if target:
                value = float(match.group("value"))
                attributes.append(
                    make_attribute(
                        target, "Organization", "market_cap_trillion_krw", value,
                        f"{match.group('value')}조 원", "number", chunk
                    )
                )

        for match in REVENUE_TARGET_PATTERN.finditer(chunk.text):
            target = find_entity_by_name(match.group("name"), chunk_entities, "Organization")
            if target:
                value = float(match.group("value"))
                attributes.append(
                    make_attribute(
                        target, "Organization", "revenue_target_trillion_krw", value,
                        f"{match.group('value')}조 원", "number", chunk
                    )
                )

        for match in STOCK_CHANGE_PATTERN.finditer(chunk.text):
            target = find_entity_by_name(match.group("name"), chunk_entities, "Organization")
            if target:
                sign = 1 if match.group("direction") == "상승" else -1
                value = sign * float(match.group("value"))
                attributes.append(
                    make_attribute(
                        target, "Organization", "stock_change_percent", value,
                        f"{match.group('value')}% {match.group('direction')}", "number", chunk
                    )
                )

        for product in [ent for ent in chunk_entities if ent.entity_type == "Product"]:
            if "AI 반도체" in chunk.text:
                attributes.append(
                    make_attribute(product, "Product", "product_category", "AI semiconductor", "AI 반도체", "string", chunk)
                )

        for match in ROLE_PATTERN.finditer(chunk.text):
            person = find_entity_by_name(match.group("name"), chunk_entities, "Person")
            if person:
                attributes.append(
                    make_attribute(person, "Person", "role", "market_analyst", match.group("role"), "string", chunk)
                )

    # 같은 target/attribute가 여러 번 잡히면 첫 evidence를 기준으로 유지한다.
    deduped: dict[tuple[str, str], AttributeCandidate] = {}
    for attr in attributes:
        deduped.setdefault((attr.target_id, attr.attribute_name), attr)
    return list(deduped.values())


attributes = extract_attributes(article, chunks, entities)
for attr in attributes:
    print(attr)


## 9. 검증 게이트

검증 게이트는 추출 결과가 그래프에 들어갈 수 있는 최소 조건을 확인한다. 객체는 온톨로지 타입이어야 하고, 관계는 domain/range를 지켜야 하며, 속성은 target 객체 타입의 스키마에 정의되어 있어야 한다. 기준선 샘플에서는 기업/제품/사람의 필수 속성도 확인한다.

**발전 가능성**

- Pydantic validation으로 필드 타입과 enum을 먼저 막고, SHACL로 그래프 수준 domain/range/cardinality를 검증할 수 있다.
- evidence verifier 모델을 추가해 관계나 속성이 실제 문장에 의해 지지되는지 binary 판정할 수 있다.
- confidence threshold, top-1/top-2 entity linking gap, unsupported relation rate를 게이트 지표로 추가할 수 있다.
- 낮은 confidence나 신규 ontology term은 자동 적재하지 않고 HITL 검토 큐로 보내는 구조가 운영에 적합하다.


In [ ]:
def validate_results(
    article: ArticleDocument,
    entities: list[EntityMention],
    relations: list[RelationCandidate],
    attributes: list[AttributeCandidate],
    ontology: dict[str, Any],
    attribute_schema: dict[str, dict[str, str]],
    required_attributes_by_type: dict[str, set[str]],
    entity_threshold: float = 0.80,
    relation_threshold: float = 0.75,
) -> dict[str, Any]:
    """객체/관계/속성이 evidence, ontology, type schema를 통과하는지 검사한다."""
    errors: list[str] = []
    allowed_entity_types = set(ontology["entity_types"])
    relation_specs = ontology["relations"]
    entity_by_id = entity_index_by_id(entities)
    type_by_id = {article.article_id: "NewsArticle", **{ent.canonical_id: ent.entity_type for ent in entity_by_id.values()}}

    if not article.text or not article.source_url or not article.published_at:
        errors.append("article is missing required canonical fields")

    for ent in entities:
        if ent.entity_type not in allowed_entity_types:
            errors.append(f"unknown entity type: {ent.entity_type}")
        if not ent.evidence_text or ent.mention_text not in ent.evidence_text:
            errors.append(f"entity evidence mismatch: {ent.canonical_id}")
        if ent.confidence < entity_threshold:
            errors.append(f"low entity confidence: {ent.canonical_id}={ent.confidence}")

    for rel in relations:
        spec = relation_specs.get(rel.predicate)
        if spec is None:
            errors.append(f"unknown relation predicate: {rel.predicate}")
            continue
        subject = entity_by_id.get(rel.subject_id)
        object_ = entity_by_id.get(rel.object_id)
        if subject is None or object_ is None:
            errors.append(f"relation refers to missing entity: {rel}")
            continue
        if subject.entity_type not in spec["domain"]:
            errors.append(f"domain violation: {rel.predicate} subject={subject.entity_type}")
        if object_.entity_type not in spec["range"]:
            errors.append(f"range violation: {rel.predicate} object={object_.entity_type}")
        if not rel.evidence_text:
            errors.append(f"missing relation evidence: {rel}")
        if rel.confidence < relation_threshold:
            errors.append(f"low relation confidence: {rel}")

    attr_names_by_target: dict[str, set[str]] = {}
    for attr in attributes:
        actual_type = type_by_id.get(attr.target_id)
        if actual_type is None:
            errors.append(f"attribute target does not exist: {attr.target_id}")
            continue
        if actual_type != attr.target_type:
            errors.append(f"attribute target type mismatch: {attr.target_id} expected={actual_type} got={attr.target_type}")
        expected_value_type = attribute_schema.get(attr.target_type, {}).get(attr.attribute_name)
        if expected_value_type is None:
            errors.append(f"attribute not allowed for type: {attr.target_type}.{attr.attribute_name}")
        elif expected_value_type != attr.value_type:
            errors.append(f"attribute value type mismatch: {attr.attribute_name} expected={expected_value_type} got={attr.value_type}")
        if not attr.evidence_text or attr.value_text == "":
            errors.append(f"invalid attribute evidence/value: {attr}")
        if attr.confidence < entity_threshold:
            errors.append(f"low attribute confidence: {attr}")
        attr_names_by_target.setdefault(attr.target_id, set()).add(attr.attribute_name)

    for target_id, target_type in type_by_id.items():
        required = required_attributes_by_type.get(target_type, set())
        missing = required - attr_names_by_target.get(target_id, set())
        if missing:
            errors.append(f"missing required attributes for {target_id}: {sorted(missing)}")

    return {
        "passed": not errors,
        "error_count": len(errors),
        "errors": errors,
        "counts": {
            "unique_entities": len(entity_by_id),
            "entity_mentions": len(entities),
            "relations": len(relations),
            "attributes": len(attributes),
        },
    }


validation_report = validate_results(
    article, entities, relations, attributes,
    ONTOLOGY, ATTRIBUTE_SCHEMA, REQUIRED_ATTRIBUTES_BY_TYPE,
)
json.dumps(validation_report, ensure_ascii=False, indent=2)


## 10. 그래프 payload와 Neo4j Cypher 생성

이 단계는 실제 DB에 쓰지 않고, Neo4j Python driver에서 바로 실행할 수 있는 parameterized Cypher 목록을 만든다. 값은 모두 parameters로 넘기고, Cypher 문자열에는 검증된 label과 relationship type만 삽입한다.

**발전 가능성**

- Neo4j에는 `UNIQUE` constraint와 index를 먼저 만들고, `MERGE` 기준 key를 명확히 해야 중복 node를 줄일 수 있다.
- 대량 기사에서는 node upsert와 relationship upsert를 batch query로 묶고 transaction 단위로 commit/rollback하는 편이 좋다.
- 실제 실행 전 `EXPLAIN`/`PROFILE` dry-run, staging database, write approval gate를 두면 잘못된 대량 write를 줄일 수 있다.
- RDF 계열 저장소도 고려한다면 같은 payload에서 RDFLib/PROV-O export를 병행할 수 있다.


In [ ]:
def relation_id(edge: dict[str, Any]) -> str:
    """동일 source/type/target/evidence 조합이 같은 relationship id를 갖도록 만든다."""
    raw = "|".join([
        edge["source"], edge["type"], edge["target"],
        edge["properties"].get("chunk_id", ""),
        edge["properties"].get("evidence_text", ""),
    ])
    return f"rel:{hashlib.sha256(raw.encode('utf-8')).hexdigest()[:16]}"


def attributes_by_target(attributes: list[AttributeCandidate]) -> dict[str, dict[str, Any]]:
    """추출된 속성을 Neo4j node property로 붙일 수 있게 target별 dict로 모은다."""
    grouped: dict[str, dict[str, Any]] = {}
    for attr in attributes:
        grouped.setdefault(attr.target_id, {})[attr.attribute_name] = attr.value
    return grouped


def build_graph_payload(
    article: ArticleDocument,
    entities: list[EntityMention],
    relations: list[RelationCandidate],
    attributes: list[AttributeCandidate],
    validation_report: dict[str, Any],
) -> dict[str, Any]:
    """검증된 객체/관계/속성을 Neo4j writer가 사용할 node/edge/provenance payload로 변환한다."""
    trace_id = f"baseline:{article.checksum}:{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"
    attr_props = attributes_by_target(attributes)

    nodes: dict[str, dict[str, Any]] = {
        article.article_id: {
            "id": article.article_id,
            "labels": ["NewsArticle"],
            "properties": {
                "id": article.article_id,
                "headline": article.headline,
                "source_url": article.source_url,
                "published_at": article.published_at,
                "checksum": article.checksum,
                "trace_id": trace_id,
                **attr_props.get(article.article_id, {}),
            },
        }
    }

    for ent in entity_index_by_id(entities).values():
        nodes[ent.canonical_id] = {
            "id": ent.canonical_id,
            "labels": [ent.entity_type],
            "properties": {
                "id": ent.canonical_id,
                "name": ent.normalized_name,
                "entity_type": ent.entity_type,
                **attr_props.get(ent.canonical_id, {}),
            },
        }

    edges: list[dict[str, Any]] = []
    for ent in entities:
        edge = {
            "source": article.article_id,
            "target": ent.canonical_id,
            "type": "MENTIONS",
            "properties": {
                "mention_text": ent.mention_text,
                "start_char": ent.start_char,
                "end_char": ent.end_char,
                "chunk_id": ent.chunk_id,
                "evidence_text": ent.evidence_text,
                "confidence": ent.confidence,
                "trace_id": trace_id,
                "source_url": article.source_url,
            },
        }
        edge["id"] = relation_id(edge)
        edge["properties"]["id"] = edge["id"]
        edges.append(edge)

    for rel in relations:
        edge = {
            "source": rel.subject_id,
            "target": rel.object_id,
            "type": rel.predicate,
            "properties": {
                "article_id": article.article_id,
                "chunk_id": rel.chunk_id,
                "evidence_text": rel.evidence_text,
                "confidence": rel.confidence,
                "trace_id": trace_id,
                "source_url": article.source_url,
            },
        }
        edge["id"] = relation_id(edge)
        edge["properties"]["id"] = edge["id"]
        edges.append(edge)

    return {
        "metadata": {
            "created_at": datetime.now(timezone.utc).isoformat(),
            "model": "rule-based-baseline-v2",
            "trace_id": trace_id,
            "validation_passed": validation_report["passed"],
        },
        "article": asdict(article),
        "nodes": list(nodes.values()),
        "edges": edges,
        "attributes": [asdict(attr) for attr in attributes],
        "validation": validation_report,
    }


def safe_cypher_name(name: str, allowed: set[str]) -> str:
    """Cypher에 직접 들어가는 label/type이 allowlist와 식별자 규칙을 통과하는지 확인한다."""
    if name not in allowed or not re.fullmatch(r"[A-ZA-Za-z_][A-ZA-Za-z0-9_]*", name):
        raise ValueError(f"unsafe cypher name: {name}")
    return f"`{name}`"


def build_cypher_queries(graph_payload: dict[str, Any], ontology: dict[str, Any]) -> list[dict[str, Any]]:
    """Neo4j driver에서 순서대로 실행할 parameterized Cypher query 목록을 만든다."""
    allowed_labels = set(ontology["entity_types"])
    allowed_relationships = set(ontology["relations"])
    queries: list[dict[str, Any]] = []

    for node in graph_payload["nodes"]:
        label = safe_cypher_name(node["labels"][0], allowed_labels)
        queries.append({
            "name": f"upsert_node_{node['id']}",
            "cypher": chr(10).join([
                f"MERGE (n:{label} {{id: $id}})",
                "SET n += $properties",
            ]),
            "parameters": {
                "id": node["id"],
                "properties": node["properties"],
            },
        })

    for edge in graph_payload["edges"]:
        rel_type = safe_cypher_name(edge["type"], allowed_relationships)
        queries.append({
            "name": f"upsert_relationship_{edge['id']}",
            "cypher": chr(10).join([
                "MATCH (source {id: $source_id})",
                "MATCH (target {id: $target_id})",
                f"MERGE (source)-[rel:{rel_type} {{id: $id}}]->(target)",
                "SET rel += $properties",
            ]),
            "parameters": {
                "source_id": edge["source"],
                "target_id": edge["target"],
                "id": edge["id"],
                "properties": edge["properties"],
            },
        })

    return queries


def validate_cypher_queries(cypher_queries: list[dict[str, Any]], graph_payload: dict[str, Any], ontology: dict[str, Any]) -> dict[str, Any]:
    """생성된 Cypher가 destructive command 없이 allowlist label/type과 parameter binding을 쓰는지 검사한다."""
    errors: list[str] = []
    expected_count = len(graph_payload["nodes"]) + len(graph_payload["edges"])
    allowed_names = set(ontology["entity_types"]) | set(ontology["relations"])
    forbidden = {"DELETE", "DETACH", "DROP", "CALL", "LOAD CSV"}

    if len(cypher_queries) != expected_count:
        errors.append(f"query count mismatch: expected={expected_count} actual={len(cypher_queries)}")

    graph_ids = {node["id"] for node in graph_payload["nodes"]} | {edge["id"] for edge in graph_payload["edges"]}
    for query in cypher_queries:
        cypher = query.get("cypher", "")
        parameters = query.get("parameters", {})
        upper = cypher.upper()
        if any(token in upper for token in forbidden):
            errors.append(f"forbidden cypher token in {query.get('name')}")
        if "$" not in cypher:
            errors.append(f"query is not parameterized: {query.get('name')}")
        if not {"id", "properties"}.issubset(parameters):
            errors.append(f"missing required parameters: {query.get('name')}")
        if any(graph_id in cypher for graph_id in graph_ids):
            errors.append(f"raw graph id interpolated into cypher: {query.get('name')}")
        for quoted_name in re.findall(r"`([^`]+)`", cypher):
            if quoted_name not in allowed_names:
                errors.append(f"cypher name is not allowlisted: {quoted_name}")
        try:
            json.dumps(parameters, ensure_ascii=False)
        except TypeError as exc:
            errors.append(f"parameters are not json serializable in {query.get('name')}: {exc}")

    return {"passed": not errors, "error_count": len(errors), "errors": errors, "query_count": len(cypher_queries)}


graph_payload = build_graph_payload(article, entities, relations, attributes, validation_report)
cypher_queries = build_cypher_queries(graph_payload, ONTOLOGY)
cypher_validation = validate_cypher_queries(cypher_queries, graph_payload, ONTOLOGY)
cypher_bundle = {"queries": cypher_queries, "validation": cypher_validation}
graph_payload["cypher"] = cypher_bundle

print(f"nodes={len(graph_payload['nodes'])}, edges={len(graph_payload['edges'])}, cypher_queries={len(cypher_queries)}")
print(json.dumps(graph_payload["metadata"], ensure_ascii=False, indent=2))
print(json.dumps(cypher_validation, ensure_ascii=False, indent=2))


## 11. 간단 평가

베이스라인은 객체, 관계, 속성을 각각 gold set과 비교한다. 실제 프로젝트에서는 기사 수를 늘리고 사람이 검수한 gold set으로 바꾸면 된다.

**발전 가능성**

- entity, relation, attribute를 타입별로 나눠 precision/recall/F1을 계산하면 어떤 단계가 약한지 빨리 보인다.
- 언론사, 언어, 산업군, 기사 길이별 slice evaluation을 만들면 도메인 이동에 따른 성능 저하를 추적할 수 있다.
- LangSmith나 자체 trace log를 연결하면 추출 실패 사례와 비용/지연 시간을 함께 분석할 수 있다.
- gold set에는 정답 triple뿐 아니라 금지해야 할 unsupported triple도 포함하는 것이 좋다.


In [ ]:
GOLD_ENTITIES = {
    ("org:samsung-electronics", "Organization"),
    ("org:sk-hynix", "Organization"),
    ("place:seoul", "Place"),
    ("product:exynos-ai", "Product"),
    ("person:kim-minsu", "Person"),
}

GOLD_RELATIONS = {
    ("org:samsung-electronics", "ANNOUNCED", "product:exynos-ai"),
    ("org:samsung-electronics", "LOCATED_IN", "place:seoul"),
    ("org:samsung-electronics", "PARTNERED_WITH", "org:sk-hynix"),
    ("org:samsung-electronics", "ANALYZED_BY", "person:kim-minsu"),
}

GOLD_ATTRIBUTES = {
    ("article", "mentioned_date"),
    ("org:samsung-electronics", "market_cap_trillion_krw"),
    ("org:sk-hynix", "market_cap_trillion_krw"),
    ("org:samsung-electronics", "revenue_target_trillion_krw"),
    ("org:samsung-electronics", "stock_change_percent"),
    ("product:exynos-ai", "product_category"),
    ("person:kim-minsu", "role"),
}


def precision_recall(predicted: set[tuple[Any, ...]], gold: set[tuple[Any, ...]]) -> dict[str, float]:
    """예측 집합과 gold 집합을 비교해 precision, recall, f1을 계산한다."""
    true_positive = len(predicted & gold)
    precision = true_positive / len(predicted) if predicted else 0.0
    recall = true_positive / len(gold) if gold else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"precision": precision, "recall": recall, "f1": f1}


pred_entities = {(ent.canonical_id, ent.entity_type) for ent in entity_index_by_id(entities).values()}
pred_relations = {(rel.subject_id, rel.predicate, rel.object_id) for rel in relations}
pred_attributes = {
    ("article" if attr.target_id == article.article_id else attr.target_id, attr.attribute_name)
    for attr in attributes
}

metrics = {
    "entity": precision_recall(pred_entities, GOLD_ENTITIES),
    "relation": precision_recall(pred_relations, GOLD_RELATIONS),
    "attribute": precision_recall(pred_attributes, GOLD_ATTRIBUTES),
    "extraction_validation_passed": validation_report["passed"],
    "cypher_validation_passed": cypher_validation["passed"],
}

json.dumps(metrics, ensure_ascii=False, indent=2)


## 12. 결과 저장

노트북 실행 결과는 `baseline/basemodel_outputs/graph_payload.json`에 저장한다. 이 파일에는 추출된 graph payload와 Neo4j 실행용 Cypher bundle이 함께 들어간다.

**발전 가능성**

- 배치 처리에서는 JSON 하나보다 article 단위 JSONL, run manifest, validation report, error report를 분리 저장하는 편이 좋다.
- `trace_id`, ontology version, extractor version, model name, prompt version을 함께 저장하면 재현성과 감사 가능성이 올라간다.
- 실패한 기사와 통과한 기사를 별도 디렉터리나 staging table로 나누면 재처리 파이프라인을 만들기 쉽다.


In [ ]:
payload_path = OUTPUT_DIR / "graph_payload.json"
with payload_path.open("w", encoding="utf-8") as f:
    json.dump(graph_payload, f, ensure_ascii=False, indent=2)

print(f"saved: {payload_path}")


## 13. 다음 단계

이 기준선은 오케스트레이션 없이도 다음을 확인한다.

- 객체 추출 결과가 canonical ID, ontology type, evidence span을 갖는다.
- 관계 추출 결과가 ontology domain/range를 지킨다.
- 객체 타입별 속성이 스키마에 맞게 추출된다.
- graph payload가 Neo4j node/relationship property로 변환된다.
- Cypher query는 값 interpolation 없이 parameterized query로 생성된다.
- destructive Cypher나 allowlist 밖 label/type은 검증 게이트에서 차단된다.

**발전 순서 제안**

1. 문장/문단 분리는 정규식에서 `kss`, spaCy sentence segmenter, LangChain splitter로 교체한다.
2. 객체 추출은 alias baseline에 spaCy NER 또는 GLiNER를 추가하고, LLM structured output은 recall 보강용으로 쓴다.
3. 관계 추출은 현재 rule을 유지한 채 spaCy dependency parsing, REBEL/mREBEL, LLM 후보 추출을 비교한다.
4. 속성 추출은 정규식에서 타입별 parser와 Pydantic schema validation으로 확장한다.
5. 검증은 Pydantic + SHACL + evidence verifier + HITL 큐로 나누어 운영 게이트로 발전시킨다.
6. Neo4j write는 지금 만든 Cypher bundle을 transaction batch writer에 연결하고 staging DB에서 먼저 검증한다.
